In [1]:
'''! pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126 --quiet
! pip install transformers --quiet
! pip install pillow --quiet
! pip install requests --quiet
! pip install tqdm --quiet
! pip install pandas --quiet
! pip install datasets --quiet
! pip install scikit-learn --quiet
! pip install numpy --quiet'''

'! pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126 --quiet\n! pip install transformers --quiet\n! pip install pillow --quiet\n! pip install requests --quiet\n! pip install tqdm --quiet\n! pip install pandas --quiet\n! pip install datasets --quiet\n! pip install scikit-learn --quiet\n! pip install numpy --quiet'

In [2]:
#!/usr/bin/env python3
"""
Import all required libraries and setup CUDA environment
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torch.backends.cudnn as cudnn
from transformers import (
    BertTokenizer, BertModel,
    RobertaTokenizer, RobertaModel,
    DistilBertTokenizer, DistilBertModel,
    CLIPProcessor, CLIPModel  # Changed from Swin
)
from PIL import Image, ImageEnhance, ImageFilter
import requests
from tqdm.auto import tqdm
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
import numpy as np
import random
import gc
import psutil
import os
warnings.filterwarnings('ignore')

# CUDA Error Fix: Set environment variables for debugging
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # For better error reporting
os.environ['TORCH_USE_CUDA_DSA'] = '1'    # Enable device-side assertions

# Enable CUDA optimizations (with error handling)
if torch.cuda.is_available():
    try:
        cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        print("✅ CUDA optimizations enabled")
    except Exception as e:
        print(f"⚠️  CUDA optimization warning: {e}")
else:
    print("⚠️  CUDA not available, using CPU")

print("📦 All libraries imported successfully!")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🔥 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🔥 GPU: {torch.cuda.get_device_name()}")

2025-10-03 15:33:00.753026: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759505580.779353     296 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759505580.786678     296 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


✅ CUDA optimizations enabled
📦 All libraries imported successfully!
🔥 PyTorch version: 2.6.0+cu124
🔥 CUDA available: True
🔥 GPU: Tesla T4


In [ ]:
"""
Configuration classes with path management and hyperparameters
"""

class PathConfig:
    """Centralized path configuration - MODIFY THESE PATHS"""
    
    # Base directory for your project
    PROJECT_DIR = r"/kaggle/input/fakeddit-captions-optimized"  # ⚠️ CHANGE THIS
    
    # Data paths
    CAPTIONS_CSV = os.path.join(PROJECT_DIR, "fakeddit_captions_optimized.csv")
    
    # Model cache directories (for downloaded BERT, RoBERTa, etc.)
    #MODEL_CACHE_DIR = r"G:\IIITK\model_data"  # ⚠️ CHANGE THIS
    PROJECT_DIR_1 = r"/kaggle/working/"
    
    # Output paths for trained models
    OUTPUT_DIR = os.path.join(PROJECT_DIR_1, "trained_models")
    BEST_MODEL_PATH = os.path.join(OUTPUT_DIR, "best_advanced_model.pth")
    
    # Logs and results
    LOGS_DIR = os.path.join(PROJECT_DIR_1, "logs")
    RESULTS_DIR = os.path.join(PROJECT_DIR_1, "results")
    
    @classmethod
    def setup_directories(cls):
        """Create necessary directories if they don't exist"""
        os.makedirs(cls.OUTPUT_DIR, exist_ok=True)
        os.makedirs(cls.LOGS_DIR, exist_ok=True) 
        os.makedirs(cls.RESULTS_DIR, exist_ok=True)
        #os.makedirs(cls.MODEL_CACHE_DIR, exist_ok=True)
        
        # Set environment variables for HuggingFace
        #os.environ["HF_HOME"] = cls.MODEL_CACHE_DIR
        #os.environ["TRANSFORMERS_CACHE"] = cls.MODEL_CACHE_DIR
        
        print(f"✅ Directories setup complete!")
        print(f"📄 CSV Path: {cls.CAPTIONS_CSV}")
        #print(f"🤖 Model Cache: {cls.MODEL_CACHE_DIR}")
        print(f"💾 Output Dir: {cls.OUTPUT_DIR}")


class Config:
    """Enhanced configuration class with path integration"""
    
    # Model parameters
    TEXT_DIM = 768
    IMG_DIM = 768
    HIDDEN_DIM = 1024
    NUM_CLASSES = 2
    DROPOUT_RATE = 0.3
    ATTENTION_HEADS = 8
    
    # Training parameters
    LEARNING_RATE = 5e-5
    WARMUP_STEPS = 1000
    BATCH_SIZE = 16  # GPU optimized
    FEATURE_BATCH_SIZE = 32
    MAX_EPOCHS = 50
    PATIENCE = 10
    GRADIENT_CLIP_VAL = 1.0
    
    # Mixed Precision Settings
    USE_MIXED_PRECISION = True
    
    # Loss function parameters
    FOCAL_ALPHA = 0.25
    FOCAL_GAMMA = 2.0
    LABEL_SMOOTHING = 0.1
    
    # Data parameters - Using PathConfig
    CAPTIONS_CSV = PathConfig.CAPTIONS_CSV
    DATASET_NAME = "rtfarchitect/fakeddit_sample"
    SAMPLE_SIZE = 100000
    TRAIN_SPLIT = 0.7
    VAL_SPLIT = 0.15
    TEST_SPLIT = 0.15
    
    
    # Model names
    BERT_MODEL = "bert-base-uncased"
    ROBERTA_MODEL = "roberta-base"
    DISTILBERT_MODEL = "distilbert-base-uncased"
    CLIP_MODEL = "openai/clip-vit-base-patch32" # Changed from SWIN_MODEL

    IMG_DIM = 512 # Changed from 768 to match CLIP's output dimension
    # File paths - Using PathConfig
    BEST_MODEL_PATH = PathConfig.BEST_MODEL_PATH
    LOGS_DIR = PathConfig.LOGS_DIR
    RESULTS_DIR = PathConfig.RESULTS_DIR
    
    # GPU Optimization settings
    NUM_WORKERS = 0  # CUDA safe
    PIN_MEMORY = False  # CUDA safe
    PREFETCH_FACTOR = None
    PERSISTENT_WORKERS = False
    EMPTY_CACHE_FREQ = 50
    GRADIENT_CHECKPOINTING = False
    
    # Data augmentation
    AUGMENT_PROB = 0.3
    
    # Other
    RANDOM_STATE = 42


# Initialize paths
PathConfig.setup_directories()

print("🔧 Configuration loaded successfully!")
print("=" * 50)
print("📋 Current Configuration:")
print(f"CSV File: {Config.CAPTIONS_CSV}")
print(f"Output Dir: {PathConfig.OUTPUT_DIR}")
print(f"Best Model: {Config.BEST_MODEL_PATH}")
print(f"Batch Size: {Config.BATCH_SIZE}")
print(f"Mixed Precision: {Config.USE_MIXED_PRECISION}")
print("=" * 50)

✅ Directories setup complete!
📄 CSV Path: /kaggle/input/fakeddit-captions-optimized/fakeddit_captions_optimized.csv
💾 Output Dir: /kaggle/working/trained_models
🔧 Configuration loaded successfully!
📋 Current Configuration:
CSV File: /kaggle/input/fakeddit-captions-optimized/fakeddit_captions_optimized.csv
Output Dir: /kaggle/working/trained_models
Best Model: /kaggle/working/trained_models/best_advanced_model.pth
Batch Size: 16
Mixed Precision: True


In [4]:
"""
Utility functions and custom loss functions
"""

def get_gpu_memory_info():
    """Get current GPU memory usage"""
    if torch.cuda.is_available():
        return {
            'allocated': torch.cuda.memory_allocated() / 1024**3,  # GB
            'cached': torch.cuda.memory_reserved() / 1024**3,      # GB
            'max_allocated': torch.cuda.max_memory_allocated() / 1024**3  # GB
        }
    return {'allocated': 0, 'cached': 0, 'max_allocated': 0}


def optimize_gpu_memory():
    """Optimize GPU memory usage"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()


class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance with mixed precision support"""
    def __init__(self, alpha=Config.FOCAL_ALPHA, gamma=Config.FOCAL_GAMMA):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1-pt)**self.gamma * ce_loss
        return focal_loss.mean()


class LabelSmoothingCrossEntropy(nn.Module):
    """Label smoothing for better generalization with mixed precision support"""
    def __init__(self, smoothing=Config.LABEL_SMOOTHING):
        super(LabelSmoothingCrossEntropy, self).__init__()
        self.smoothing = smoothing

    def forward(self, x, target):
        confidence = 1. - self.smoothing
        logprobs = F.log_softmax(x, dim=-1)
        nll_loss = -logprobs.gather(dim=-1, index=target.unsqueeze(1))
        nll_loss = nll_loss.squeeze(1)
        smooth_loss = -logprobs.mean(dim=-1)
        loss = confidence * nll_loss + self.smoothing * smooth_loss
        return loss.mean()


def setup_gpu_environment():
    """Setup optimal GPU environment"""
    if torch.cuda.is_available():
        print(f"🚀 GPU Device: {torch.cuda.get_device_name()}")
        print(f"🚀 CUDA Version: {torch.version.cuda}")
        print(f"🚀 PyTorch Version: {torch.__version__}")
        print(f"🚀 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB")
        
        # Enable optimizations
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        
        print("✅ GPU optimizations enabled")
        return True
    else:
        print("⚠️  CUDA not available, using CPU")
        return False


print("🛠️ Utility functions and loss classes defined!")
print(f"🚀 Current GPU Memory: {get_gpu_memory_info()}")

🛠️ Utility functions and loss classes defined!
🚀 Current GPU Memory: {'allocated': 0.0, 'cached': 0.0, 'max_allocated': 0.0}


In [5]:
"""
Advanced model architecture with cross-modal attention and multiple fusion strategies
"""

class MultiHeadCrossModalAttention(nn.Module):
    """Multi-head cross-modal attention mechanism with gradient checkpointing"""
    def __init__(self, d_model=Config.TEXT_DIM, n_heads=Config.ATTENTION_HEADS):
        super(MultiHeadCrossModalAttention, self).__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)
        self.output = nn.Linear(d_model, d_model)
        
    def forward(self, text_emb, img_emb):
        batch_size = text_emb.size(0)
        
        # Multi-head attention
        Q = self.query(text_emb).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.key(img_emb).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.value(img_emb).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        attention_weights = F.softmax(scores, dim=-1)
        context = torch.matmul(attention_weights, V)
        
        # Concatenate heads and put through final linear layer
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, Config.TEXT_DIM)
        output = self.output(context.squeeze(1))
        
        return output, attention_weights


class AdvancedMultimodalClassifier(nn.Module):
    """
    Advanced multimodal classifier with ensemble of BERT variants,
    cross-modal attention, and multiple fusion strategies.
    Optimized for mixed precision training.
    """
    
    def __init__(self, text_dim=Config.TEXT_DIM, img_dim=Config.IMG_DIM, 
                 hidden_dim=Config.HIDDEN_DIM, num_classes=Config.NUM_CLASSES):
        super(AdvancedMultimodalClassifier, self).__init__()
        
        # Multiple fusion strategies
        # Early fusion
        self.early_fusion = nn.Sequential(
            nn.Linear(text_dim + img_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT_RATE),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT_RATE)
        )

        # ADD THIS LINE to project image embedding to text embedding size
        self.img_projection = nn.Linear(img_dim, text_dim)
        # Cross-modal attention
        self.cross_attention = MultiHeadCrossModalAttention()
        
        # Middle fusion
        self.middle_fusion = nn.Sequential(
            nn.Linear(text_dim * 2, hidden_dim),  # text + attended features
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT_RATE),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT_RATE)
        )
        
        # Late fusion - separate processing
        self.text_branch = nn.Sequential(
            nn.Linear(text_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT_RATE),
            nn.Linear(hidden_dim // 2, hidden_dim // 4)
        )
        
        self.img_branch = nn.Sequential(
            nn.Linear(img_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT_RATE),
            nn.Linear(hidden_dim // 2, hidden_dim // 4)
        )
        
        # Fusion weights (learnable)
        self.fusion_weights = nn.Parameter(torch.ones(3) / 3)  # early, middle, late
        
        # Final classification layers
        total_features = (hidden_dim // 2) * 3  # from all three fusion strategies
        self.classifier = nn.Sequential(
            nn.Linear(total_features, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT_RATE),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT_RATE),
            nn.Linear(hidden_dim // 2, num_classes)
        )
        
        # Consistency checker
        self.consistency_checker = nn.Sequential(
            nn.Linear(text_dim + img_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
        
        # Initialize weights properly for mixed precision
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        """Initialize weights for better mixed precision performance"""
        if isinstance(module, nn.Linear):
            torch.nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.zeros_(module.bias)
            torch.nn.init.ones_(module.weight)

    def forward(self, text_emb, img_emb):
        """Advanced forward pass with multiple fusion strategies"""
        
        # Early fusion
        early_concat = torch.cat((text_emb, img_emb), dim=1)
        early_features = self.early_fusion(early_concat)
        
        # Cross-modal attention for middle fusion
        #attended_features, attention_weights = self.cross_attention(text_emb, img_emb)

        img_emb_projected = self.img_projection(img_emb)
        attended_features, attention_weights = self.cross_attention(text_emb, img_emb_projected)
        
        middle_concat = torch.cat((text_emb, attended_features), dim=1)
        middle_features = self.middle_fusion(middle_concat)
        
        # Late fusion - separate processing
        text_features = self.text_branch(text_emb)
        img_features = self.img_branch(img_emb)
        late_features = torch.cat((text_features, img_features), dim=1)
        
        # Weighted fusion of all strategies
        fusion_weights_normalized = F.softmax(self.fusion_weights, dim=0)
        combined_features = torch.cat([
            early_features * fusion_weights_normalized[0],
            middle_features * fusion_weights_normalized[1],
            late_features * fusion_weights_normalized[2]
        ], dim=1)
        
        # Final classification
        logits = self.classifier(combined_features)
        
        # Consistency score (for additional loss)
        consistency_score = self.consistency_checker(early_concat)
        
        return logits, consistency_score, attention_weights


print("🧠 Advanced model architecture classes defined!")
print("✅ Multi-head cross-modal attention implemented")
print("✅ Triple fusion strategy (early, middle, late) implemented")
print("✅ Consistency checker for multimodal alignment")

🧠 Advanced model architecture classes defined!
✅ Multi-head cross-modal attention implemented
✅ Triple fusion strategy (early, middle, late) implemented
✅ Consistency checker for multimodal alignment


In [6]:
"""
GPU-optimized feature extraction with multi-BERT ensemble
"""

class GPUOptimizedFeatureExtractor:
    """
    GPU-optimized feature extraction with batch processing and mixed precision support.
    """
    
    def __init__(self, device):
        self.device = device
        self._load_models()
        print(f"🚀 GPU Memory after model loading: {get_gpu_memory_info()}")
    
    def _load_models(self):
        """Load multiple pre-trained models for ensemble with GPU optimization"""
        print("Loading GPU-optimized ensemble models...")
        
        # BERT
        self.bert_tokenizer = BertTokenizer.from_pretrained(Config.BERT_MODEL)
        self.bert_model = BertModel.from_pretrained(Config.BERT_MODEL).to(self.device)
        self.bert_model.eval()  # Set to eval mode for feature extraction
        
        # RoBERTa
        self.roberta_tokenizer = RobertaTokenizer.from_pretrained(Config.ROBERTA_MODEL)
        self.roberta_model = RobertaModel.from_pretrained(Config.ROBERTA_MODEL).to(self.device)
        self.roberta_model.eval()
        
        # DistilBERT
        self.distilbert_tokenizer = DistilBertTokenizer.from_pretrained(Config.DISTILBERT_MODEL)
        self.distilbert_model = DistilBertModel.from_pretrained(Config.DISTILBERT_MODEL).to(self.device)
        self.distilbert_model.eval()
        
        # CLIP for images (Changed from Swin/ViT)
        self.clip_processor = CLIPProcessor.from_pretrained(Config.CLIP_MODEL)
        self.clip_model = CLIPModel.from_pretrained(Config.CLIP_MODEL).to(self.device)
        self.clip_model.eval()

        # Enable mixed precision inference
        if Config.USE_MIXED_PRECISION:
            self.bert_model = self.bert_model.half()
            self.roberta_model = self.roberta_model.half()
            self.distilbert_model = self.distilbert_model.half()
            self.clip_model = self.clip_model.half() # Changed from swin_model
            print("✅ Models converted to FP16 for mixed precision")

        print("✅ GPU-optimized ensemble models loaded successfully!")
    
    def augment_text_batch(self, texts):
        """Batch text augmentation for GPU efficiency"""
        if random.random() < Config.AUGMENT_PROB:
            augmented = []
            for text in texts:
                words = text.split()
                if len(words) > 3 and random.random() < 0.5:
                    dropout_idx = random.randint(0, len(words) - 1)
                    words.pop(dropout_idx)
                    augmented.append(' '.join(words))
                else:
                    augmented.append(text)
            return augmented
        return texts
    
    def augment_images_batch(self, images):
        """Batch image augmentation"""
        if random.random() < Config.AUGMENT_PROB:
            augmented = []
            for image in images:
                if random.random() < 0.5:
                    try:
                        aug_funcs = [
                            lambda img: ImageEnhance.Brightness(img).enhance(random.uniform(0.8, 1.2)),
                            lambda img: ImageEnhance.Contrast(img).enhance(random.uniform(0.8, 1.2)),
                            lambda img: img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0, 0.5))),
                        ]
                        aug_func = random.choice(aug_funcs)
                        augmented.append(aug_func(image))
                    except:
                        augmented.append(image)
                else:
                    augmented.append(image)
            return augmented
        return images
    
    def _extract_bert_batch(self, batch_texts):
        """Extract BERT embeddings for a batch"""
        inputs = self.bert_tokenizer(batch_texts, return_tensors="pt", 
                                   truncation=True, padding=True, 
                                   max_length=512).to(self.device)
        outputs = self.bert_model(**inputs)
        return outputs.last_hidden_state[:, 0, :].clone().detach()
    
    def _extract_roberta_batch(self, batch_texts):
        """Extract RoBERTa embeddings for a batch"""
        inputs = self.roberta_tokenizer(batch_texts, return_tensors="pt", 
                                      truncation=True, padding=True, 
                                      max_length=512).to(self.device)
        outputs = self.roberta_model(**inputs)
        return outputs.last_hidden_state[:, 0, :].clone().detach()
    
    def _extract_distilbert_batch(self, batch_texts):
        """Extract DistilBERT embeddings for a batch"""
        inputs = self.distilbert_tokenizer(batch_texts, return_tensors="pt", 
                                         truncation=True, padding=True, 
                                         max_length=512).to(self.device)
        outputs = self.distilbert_model(**inputs)
        return outputs.last_hidden_state[:, 0, :].clone().detach()


print("🔍 Feature extraction classes defined!")
print("✅ Multi-BERT ensemble extractor (BERT + RoBERTa + DistilBERT)")
print("✅ GPU-optimized batch processing")
print("✅ Mixed precision support")
print("✅ Data augmentation capabilities")

🔍 Feature extraction classes defined!
✅ Multi-BERT ensemble extractor (BERT + RoBERTa + DistilBERT)
✅ GPU-optimized batch processing
✅ Mixed precision support
✅ Data augmentation capabilities


In [7]:
"""
Data loading, preprocessing, and management
This cell is separate so you can modify data loading without reloading models
"""

class AdvancedDataManager:
    """Enhanced data manager with GPU-optimized preprocessing"""
    
    def __init__(self, feature_extractor):
        self.feature_extractor = feature_extractor
        self.df = None
        self.text_embs = None
        self.img_embs = None
    
    def load_and_merge_datasets(self, captions_csv_path=None):
        """Load and merge datasets with enhanced preprocessing"""
        if captions_csv_path is None:
            captions_csv_path = Config.CAPTIONS_CSV
            
        try:
            print(f"📊 Loading Fakeddit dataset: {Config.DATASET_NAME}")
            hf_dataset = load_dataset(Config.DATASET_NAME)
            print(f"✅ Loaded HuggingFace dataset: {hf_dataset}")
            
            train_data = hf_dataset['train']
            
            if Config.SAMPLE_SIZE and Config.SAMPLE_SIZE < len(train_data):
                indices = list(range(Config.SAMPLE_SIZE))
                train_data = train_data.select(indices)
            
            fakeddit_df = pd.DataFrame(train_data[:])
            print(f"Fakeddit dataset shape: {fakeddit_df.shape}")
            
            print(f"📄 Loading captions CSV: {captions_csv_path}")
            captions_df = pd.read_csv(captions_csv_path)
            print(f"✅ Loaded captions CSV")
            print(f"Captions CSV shape: {captions_df.shape}")
            
            print("🔗 Merging datasets...")
            min_len = min(len(fakeddit_df), len(captions_df))
            if len(fakeddit_df) != len(captions_df):
                print(f"⚠️  Dataset size mismatch! Using first {min_len} records from both")
                fakeddit_df = fakeddit_df.head(min_len)
                captions_df = captions_df.head(min_len)
            
            self.df = pd.DataFrame({
                'title': fakeddit_df['title'].values,
                'image_url': fakeddit_df['image_url'].values, 
                '2_way_label': fakeddit_df['2_way_label'].values,
                'caption': captions_df['caption'].values
            })
            
            print(f"✅ Merged dataset created!")
            print(f"Final dataset shape: {self.df.shape}")
            
            return True
            
        except Exception as e:
            print(f"❌ Dataset loading/merging failed: {e}")
            return False
    
    def prepare_data(self):
        """Enhanced data preparation with cleaning and balancing"""
        if self.df is None:
            raise ValueError("Dataset not loaded. Call load_and_merge_datasets() first.")
        
        print(f"Using all {len(self.df)} records for training")
        
        # Enhanced data cleaning
        self.df['title'] = self.df['title'].fillna('').astype(str)
        self.df['caption'] = self.df['caption'].fillna('').astype(str)
        
        # Remove very short texts
        original_len = len(self.df)
        self.df = self.df[self.df['title'].str.len() > 10]
        print(f"Removed {original_len - len(self.df)} samples with short titles")
        
        print("✅ Enhanced data preparation complete!")
        print(f"Label distribution:")
        print(self.df['2_way_label'].value_counts())
        print(f"Final cleaned dataset size: {len(self.df)}")
    
    def split_data(self):
        """Stratified data splitting with GPU optimization"""
        if self.text_embs is None or self.img_embs is None:
            raise ValueError("Features not extracted. Call extract_features() first.")
        
        indices = list(range(len(self.df)))
        
        train_idx, temp_idx = train_test_split(
            indices, test_size=0.3, random_state=Config.RANDOM_STATE,
            stratify=self.df['2_way_label']
        )
        
        val_idx, test_idx = train_test_split(
            temp_idx, test_size=0.5, random_state=Config.RANDOM_STATE,
            stratify=self.df.iloc[temp_idx]['2_way_label']
        )
        
        print(f"✅ Train: {len(train_idx)} samples")
        print(f"✅ Validation: {len(val_idx)} samples")
        print(f"✅ Test: {len(test_idx)} samples")
        
        return train_idx, val_idx, test_idx
    
    def create_dataloaders(self, train_idx, val_idx, test_idx, device):
        """Create GPU-optimized DataLoaders with advanced settings"""
        all_labels = torch.tensor(self.df['2_way_label'].values).to(device)
        
        # Split tensors
        train_text_embs = self.text_embs[train_idx]
        train_img_embs = self.img_embs[train_idx]
        train_labels = all_labels[train_idx]
        
        val_text_embs = self.text_embs[val_idx]
        val_img_embs = self.img_embs[val_idx]
        val_labels = all_labels[val_idx]
        
        test_text_embs = self.text_embs[test_idx]
        test_img_embs = self.img_embs[test_idx]
        test_labels = all_labels[test_idx]
        
        # Create datasets
        train_dataset = TensorDataset(train_text_embs, train_img_embs, train_labels)
        val_dataset = TensorDataset(val_text_embs, val_img_embs, val_labels)
        test_dataset = TensorDataset(test_text_embs, test_img_embs, test_labels)
        
        # Create GPU-optimized dataloaders (CUDA-safe settings)
        train_loader = DataLoader(
            train_dataset, 
            batch_size=Config.BATCH_SIZE, 
            shuffle=True, 
            num_workers=Config.NUM_WORKERS,  # 0 for CUDA safety
            pin_memory=Config.PIN_MEMORY     # False for CUDA safety
        )
        
        val_loader = DataLoader(
            val_dataset, 
            batch_size=Config.BATCH_SIZE, 
            shuffle=False, 
            num_workers=Config.NUM_WORKERS,
            pin_memory=Config.PIN_MEMORY
        )
        
        test_loader = DataLoader(
            test_dataset, 
            batch_size=Config.BATCH_SIZE, 
            shuffle=False, 
            num_workers=Config.NUM_WORKERS,
            pin_memory=Config.PIN_MEMORY
        )
        
        print("✅ CUDA-safe DataLoaders created!")
        return train_loader, val_loader, test_loader


print("📊 Data management classes defined!")
print("✅ Hybrid data loading (HuggingFace + CSV)")
print("✅ Intelligent data cleaning and preprocessing")
print("✅ Stratified data splitting")
print("✅ CUDA-safe DataLoader creation")

📊 Data management classes defined!
✅ Hybrid data loading (HuggingFace + CSV)
✅ Intelligent data cleaning and preprocessing
✅ Stratified data splitting
✅ CUDA-safe DataLoader creation


In [8]:
"""
Add feature extraction methods to the GPUOptimizedFeatureExtractor class
This is separate so you can run it independently after loading models
"""

def extract_ensemble_text_embeddings(self, titles, captions):
    """Extract embeddings using ensemble of BERT variants with GPU optimization"""
    print("🔥 Starting GPU-optimized ensemble text embedding extraction...")
    
    all_texts = []
    for title, caption in zip(titles, captions):
        if pd.isna(caption) or caption is None or caption == "":
            caption = ""
        text = str(title) + " " + str(caption)
        all_texts.append(text)
    
    # Process in batches for GPU efficiency
    text_embs = []
    batch_size = Config.FEATURE_BATCH_SIZE
    
    with torch.no_grad():
        for i in tqdm(range(0, len(all_texts), batch_size), desc="GPU Batch Text Processing"):
            batch_texts = all_texts[i:i + batch_size]
            batch_texts = self.augment_text_batch(batch_texts)
            
            # Extract features from all three models for the batch
            if Config.USE_MIXED_PRECISION:
                with autocast():
                    bert_batch = self._extract_bert_batch(batch_texts)
                    roberta_batch = self._extract_roberta_batch(batch_texts)
                    distilbert_batch = self._extract_distilbert_batch(batch_texts)
            else:
                bert_batch = self._extract_bert_batch(batch_texts)
                roberta_batch = self._extract_roberta_batch(batch_texts)
                distilbert_batch = self._extract_distilbert_batch(batch_texts)
            
            # Ensemble averaging
            ensemble_batch = (bert_batch + roberta_batch + distilbert_batch) / 3.0
            text_embs.append(ensemble_batch)
            
            # Clear cache periodically
            if i % (Config.EMPTY_CACHE_FREQ * batch_size) == 0:
                optimize_gpu_memory()
    
    final_embeddings = torch.cat(text_embs, dim=0)
    print(f"🎯 Final text embeddings shape: {final_embeddings.shape}")
    print(f"🚀 GPU Memory after text extraction: {get_gpu_memory_info()}")
    
    return final_embeddings


def extract_enhanced_image_embeddings(self, image_urls):
    """Extract enhanced ViT embeddings with GPU-optimized batch processing"""
    print("🔥 Starting GPU-optimized image embedding extraction...")
    
    img_embs = []
    failed_count = 0
    batch_size = Config.FEATURE_BATCH_SIZE // 4  # Smaller batch for images
    
    with torch.no_grad():
        for i in tqdm(range(0, len(image_urls), batch_size), desc="GPU Batch Image Processing"):
            batch_urls = image_urls[i:i + batch_size]
            batch_images = []
            batch_valid_indices = []
            
            # Load batch of images
            for j, url in enumerate(batch_urls):
                try:
                    if pd.isna(url) or url is None:
                        continue
                    
                    response = requests.get(url, stream=True, timeout=10)
                    image = Image.open(response.raw).convert("RGB")
                    batch_images.append(image)
                    batch_valid_indices.append(j)
                except:
                    failed_count += 1
                    continue
            
            if batch_images:
                # Apply augmentation
                batch_images = self.augment_images_batch(batch_images)
                
                # Process batch with ViT
                inputs = self.clip_processor(images=batch_images, return_tensors="pt").to(self.device)
                
                if Config.USE_MIXED_PRECISION:
                    with autocast():
                        # Get image features from CLIP model
                        image_features = self.clip_model.get_image_features(pixel_values=inputs.pixel_values)
                        batch_embs = image_features.clone().detach()
                else:
                    # Get image features from CLIP model
                    image_features = self.clip_model.get_image_features(pixel_values=inputs.pixel_values)
                    batch_embs = image_features.clone().detach()
                
                # Reconstruct full batch with zero embeddings for failed images
                full_batch_embs = []
                batch_emb_idx = 0
                for j in range(len(batch_urls)):
                    if j in batch_valid_indices:
                        full_batch_embs.append(batch_embs[batch_emb_idx])
                        batch_emb_idx += 1
                    else:
                        full_batch_embs.append(torch.zeros(Config.IMG_DIM).to(self.device))
                
                img_embs.extend(full_batch_embs)
            else:
                # All images in batch failed
                for _ in range(len(batch_urls)):
                    img_embs.append(torch.zeros(Config.IMG_DIM).to(self.device))
            
            # Clear cache periodically
            if i % (Config.EMPTY_CACHE_FREQ * batch_size) == 0:
                optimize_gpu_memory()
    
    # Handle any remaining URLs not processed in batches
    remaining = len(image_urls) - len(img_embs)
    for _ in range(remaining):
        img_embs.append(torch.zeros(Config.IMG_DIM).to(self.device))
        failed_count += 1
    
    final_embeddings = torch.stack(img_embs)
    print(f"⚠️  {failed_count}/{len(image_urls)} images failed to load (using zero embeddings)")
    print(f"🎯 Final image embeddings shape: {final_embeddings.shape}")
    print(f"🚀 GPU Memory after image extraction: {get_gpu_memory_info()}")
    
    return final_embeddings


def extract_features_for_data_manager(self):
    """Extract features using GPU-optimized ensemble methods"""
    if self.df is None:
        raise ValueError("Data not prepared. Call prepare_data() first.")
    
    print("🔥 Starting GPU-optimized feature extraction...")
    print(f"🚀 Initial GPU Memory: {get_gpu_memory_info()}")
    
    # Extract text embeddings
    self.text_embs = self.feature_extractor.extract_ensemble_text_embeddings(
        self.df['title'], self.df['caption']
    )
    
    # Extract image embeddings
    self.img_embs = self.feature_extractor.extract_enhanced_image_embeddings(
        self.df['image_url']
    )
    
    print("✅ GPU-optimized feature extraction complete!")
    print(f"Text embeddings shape: {self.text_embs.shape}")
    print(f"Image embeddings shape: {self.img_embs.shape}")
    print(f"🚀 Final GPU Memory: {get_gpu_memory_info()}")
    
    # Final memory cleanup
    optimize_gpu_memory()


# Add methods to classes
GPUOptimizedFeatureExtractor.extract_ensemble_text_embeddings = extract_ensemble_text_embeddings
GPUOptimizedFeatureExtractor.extract_enhanced_image_embeddings = extract_enhanced_image_embeddings
AdvancedDataManager.extract_features = extract_features_for_data_manager

print("🔥 Feature extraction methods added to classes!")
print("✅ Multi-BERT ensemble text extraction")
print("✅ GPU-optimized image extraction with Swin Transformer")
print("✅ Batch processing with memory management")
print("✅ Mixed precision support")

🔥 Feature extraction methods added to classes!
✅ Multi-BERT ensemble text extraction
✅ GPU-optimized image extraction with Swin Transformer
✅ Batch processing with memory management
✅ Mixed precision support


In [9]:
"""
Advanced training with mixed precision and comprehensive evaluation
"""

class AdvancedTrainer:
    """Advanced training with mixed precision and GPU optimization"""
    
    def __init__(self, model, device):
        self.model = model
        self.device = device
        
        # Advanced loss functions
        self.focal_loss = FocalLoss()
        self.label_smooth_loss = LabelSmoothingCrossEntropy()
        
        # Advanced optimizer with scheduling
        self.optimizer = optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE, 
                                   weight_decay=0.01, eps=1e-8)
        
        # Learning rate scheduler
        self.scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            self.optimizer, T_0=10, eta_min=1e-7
        )
        
        # Mixed Precision Training
        self.scaler = GradScaler() if Config.USE_MIXED_PRECISION else None
        
        self.best_val_loss = float('inf')
        self.trigger_times = 0
        
        print(f"🔥 Trainer initialized with mixed precision: {Config.USE_MIXED_PRECISION}")
        print(f"🚀 GPU Memory after trainer init: {get_gpu_memory_info()}")
    
    def train_epoch(self, train_loader):
        """Advanced training epoch with mixed precision and GPU optimization"""
        self.model.train()
        total_train_loss = 0
        train_loop = tqdm(train_loader, desc="🔥 Mixed Precision Training")
        
        for batch_idx, (text_batch, img_batch, label_batch) in enumerate(train_loop):
            self.optimizer.zero_grad()
            
            if Config.USE_MIXED_PRECISION:
                # Mixed precision forward pass
                with autocast():
                    logits, consistency_score, attention_weights = self.model(text_batch, img_batch)
                    
                    # Multiple loss components
                    focal_loss = self.focal_loss(logits, label_batch)
                    smooth_loss = self.label_smooth_loss(logits, label_batch)
                    
                    # Consistency loss (encourage text-image consistency for real news)
                    consistency_target = label_batch.float()
                    consistency_loss = F.binary_cross_entropy(consistency_score.squeeze(), consistency_target)
                    
                    # Combined loss
                    total_loss = 0.5 * focal_loss + 0.3 * smooth_loss + 0.2 * consistency_loss
                
                # Mixed precision backward pass
                self.scaler.scale(total_loss).backward()
                
                # Unscale before gradient clipping
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), Config.GRADIENT_CLIP_VAL)
                
                # Optimizer step with scaling
                self.scaler.step(self.optimizer)
                self.scaler.update()
                
            else:
                # Standard precision training
                logits, consistency_score, attention_weights = self.model(text_batch, img_batch)
                
                focal_loss = self.focal_loss(logits, label_batch)
                smooth_loss = self.label_smooth_loss(logits, label_batch)
                consistency_target = label_batch.float()
                consistency_loss = F.binary_cross_entropy(consistency_score.squeeze(), consistency_target)
                
                total_loss = 0.5 * focal_loss + 0.3 * smooth_loss + 0.2 * consistency_loss
                total_loss.backward()
                
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), Config.GRADIENT_CLIP_VAL)
                self.optimizer.step()
            
            self.scheduler.step()
            
            total_train_loss += total_loss.item()
            
            # Memory management
            if batch_idx % Config.EMPTY_CACHE_FREQ == 0:
                optimize_gpu_memory()
            
            # Update progress bar
            train_loop.set_postfix({
                'loss': total_train_loss / (batch_idx + 1),
                'lr': self.scheduler.get_last_lr()[0],
                'gpu_mem': f"{get_gpu_memory_info()['allocated']:.1f}GB"
            })
        
        return total_train_loss / len(train_loader)
    
    def validate(self, val_loader):
        """Advanced validation with mixed precision"""
        self.model.eval()
        total_val_loss = 0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for text_batch, img_batch, label_batch in tqdm(val_loader, desc="🔍 Validation"):
                if Config.USE_MIXED_PRECISION:
                    with autocast():
                        logits, consistency_score, attention_weights = self.model(text_batch, img_batch)
                        loss = self.focal_loss(logits, label_batch)
                else:
                    logits, consistency_score, attention_weights = self.model(text_batch, img_batch)
                    loss = self.focal_loss(logits, label_batch)
                
                total_val_loss += loss.item()
                
                _, predicted = torch.max(logits, 1)
                total += label_batch.size(0)
                correct += (predicted == label_batch).sum().item()
        
        avg_val_loss = total_val_loss / len(val_loader)
        val_acc = 100 * correct / total
        
        return avg_val_loss, val_acc
    
    def train(self, train_loader, val_loader):
        """Advanced training loop with mixed precision and GPU optimization"""
        print("🔥 Starting GPU-optimized mixed precision training...")
        print(f"🚀 Training GPU Memory: {get_gpu_memory_info()}")
        
        for epoch in range(Config.MAX_EPOCHS):
            # Training phase
            avg_train_loss = self.train_epoch(train_loader)
            
            # Validation phase
            avg_val_loss, val_acc = self.validate(val_loader)
            
            print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | "
                  f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}% | "
                  f"LR: {self.scheduler.get_last_lr()[0]:.2e} | "
                  f"GPU: {get_gpu_memory_info()['allocated']:.1f}GB")
            
            # Enhanced early stopping with mixed precision checkpoint
            if avg_val_loss < self.best_val_loss:
                self.best_val_loss = avg_val_loss
                self.trigger_times = 0
                
                # Save comprehensive checkpoint
                checkpoint = {
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'scheduler_state_dict': self.scheduler.state_dict(),
                    'epoch': epoch,
                    'val_loss': avg_val_loss,
                    'val_acc': val_acc,
                    'mixed_precision': Config.USE_MIXED_PRECISION
                }
                
                if self.scaler:
                    checkpoint['scaler_state_dict'] = self.scaler.state_dict()
                
                torch.save(checkpoint, Config.BEST_MODEL_PATH)
                print("✅ Saved best GPU-optimized model with mixed precision")
            else:
                self.trigger_times += 1
                if self.trigger_times >= Config.PATIENCE:
                    print(f"🛑 Early stopping at epoch {epoch+1}")
                    break
            
            # Periodic memory cleanup
            optimize_gpu_memory()
        
        print("✅ GPU-optimized mixed precision training completed!")
        print(f"🚀 Final GPU Memory: {get_gpu_memory_info()}")


class AdvancedEvaluator:
    """Advanced evaluation with mixed precision support and comprehensive metrics"""
    
    def __init__(self, model, device):
        self.model = model
        self.device = device
    
    def evaluate(self, test_loader):
        """Comprehensive evaluation with mixed precision support"""
        print("🔥 Starting GPU-optimized evaluation...")
        
        # Load best model checkpoint
        checkpoint = torch.load(Config.BEST_MODEL_PATH)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.eval()
        
        test_preds = []
        test_labels_list = []
        attention_weights_list = []
        
        with torch.no_grad():
            for text_batch, img_batch, label_batch in tqdm(test_loader, desc="🎯 GPU Test Evaluation"):
                if Config.USE_MIXED_PRECISION:
                    with autocast():
                        logits, consistency_score, attention_weights = self.model(text_batch, img_batch)
                else:
                    logits, consistency_score, attention_weights = self.model(text_batch, img_batch)
                
                _, predicted = torch.max(logits, 1)
                test_preds.extend(predicted.cpu().numpy())
                test_labels_list.extend(label_batch.cpu().numpy())
                attention_weights_list.append(attention_weights.cpu())
        
        # Calculate metrics
        test_acc = accuracy_score(test_labels_list, test_preds)
        
        print(f"🎯 FINAL GPU-OPTIMIZED TEST ACCURACY: {test_acc:.4f} ({test_acc*100:.2f}%)")
        print(f"📊 Model checkpoint info:")
        print(f"   • Epoch: {checkpoint['epoch']}")
        print(f"   • Val Acc: {checkpoint['val_acc']:.2f}%") 
        print(f"   • Mixed Precision: {checkpoint['mixed_precision']}")
        print(f"🚀 Final GPU Memory: {get_gpu_memory_info()}")
        print("\\n📋 Advanced Classification Report:")
        print(classification_report(test_labels_list, test_preds, target_names=["Real", "Fake"]))
        print("\\n🧮 Confusion Matrix:")
        print(confusion_matrix(test_labels_list, test_preds))
        
        return test_acc, test_preds, test_labels_list


print("🎯 Training and evaluation classes defined!")
print("✅ Advanced trainer with mixed precision support")
print("✅ Multiple loss functions (Focal + Label Smoothing + Consistency)")
print("✅ Advanced optimizer with cosine annealing")
print("✅ Comprehensive evaluator with detailed metrics")
print("✅ Early stopping with model checkpointing")

🎯 Training and evaluation classes defined!
✅ Advanced trainer with mixed precision support
✅ Multiple loss functions (Focal + Label Smoothing + Consistency)
✅ Advanced optimizer with cosine annealing
✅ Comprehensive evaluator with detailed metrics
✅ Early stopping with model checkpointing


In [10]:
"""
Initialize the complete system - GPU setup and model loading
Run this cell first to set up everything!
"""

# Setup environment
print("🚀 Initializing Advanced Multimodal Fake News Detection System")
print("=" * 70)
print("🔥 Features:")
print("   • Mixed Precision Training (FP16) - 2x Speed & 50% Memory")
print("   • Multi-BERT Ensemble (BERT + RoBERTa + DistilBERT)")
print("   • Cross-modal Attention Mechanisms")
print("   • Multiple Fusion Strategies")
print("   • Advanced Loss Functions & Optimization")
print("=" * 70)

# Setup GPU environment
cuda_available = setup_gpu_environment()
device = torch.device("cuda" if cuda_available else "cpu")
print(f"\n🎯 Using device: {device}")

# Set seeds for reproducibility
torch.manual_seed(Config.RANDOM_STATE)
np.random.seed(Config.RANDOM_STATE)
random.seed(Config.RANDOM_STATE)
if cuda_available:
    torch.cuda.manual_seed(Config.RANDOM_STATE)

print("✅ Random seeds set for reproducibility")

# Initialize GPU-optimized components
print("\n🔥 Loading models (this may take a few minutes)...")
try:
    feature_extractor = GPUOptimizedFeatureExtractor(device)
    data_manager = AdvancedDataManager(feature_extractor)
    print("✅ All components initialized successfully!")
    
    # Show current memory usage
    print(f"🚀 Current GPU Memory: {get_gpu_memory_info()}")
    
except Exception as e:
    print(f"❌ Initialization error: {e}")
    print("\n🔧 Try these fixes:")
    print("1. Restart notebook kernel")
    print("2. Reduce Config.BATCH_SIZE to 8")
    print("3. Set Config.USE_MIXED_PRECISION = False")
    raise

print("\n🎉 System initialization complete! You can now run the next cells.")
print("📋 Next steps:")
print("   1. Run Cell 10 to load and prepare data")
print("   2. Run Cell 11 to extract features (GPU intensive!)")
print("   3. Run Cell 12 to train the model")
print("   4. Run Cell 13 to evaluate results")

🚀 Initializing Advanced Multimodal Fake News Detection System
🔥 Features:
   • Mixed Precision Training (FP16) - 2x Speed & 50% Memory
   • Multi-BERT Ensemble (BERT + RoBERTa + DistilBERT)
   • Cross-modal Attention Mechanisms
   • Multiple Fusion Strategies
   • Advanced Loss Functions & Optimization
🚀 GPU Device: Tesla T4
🚀 CUDA Version: 12.4
🚀 PyTorch Version: 2.6.0+cu124
🚀 GPU Memory: 14.7GB
✅ GPU optimizations enabled

🎯 Using device: cuda
✅ Random seeds set for reproducibility

🔥 Loading models (this may take a few minutes)...
Loading GPU-optimized ensemble models...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✅ Models converted to FP16 for mixed precision
✅ GPU-optimized ensemble models loaded successfully!
🚀 GPU Memory after model loading: {'allocated': 0.8871989250183105, 'cached': 1.86328125, 'max_allocated': 1.7311859130859375}
✅ All components initialized successfully!
🚀 Current GPU Memory: {'allocated': 0.8871989250183105, 'cached': 1.86328125, 'max_allocated': 1.7311859130859375}

🎉 System initialization complete! You can now run the next cells.
📋 Next steps:
   1. Run Cell 10 to load and prepare data
   2. Run Cell 11 to extract features (GPU intensive!)
   3. Run Cell 12 to train the model
   4. Run Cell 13 to evaluate results


In [11]:
"""
Load and prepare datasets - this cell can be run independently
Modify paths here if needed!
"""

print("📊 Starting data loading and preparation...")
print(f"📄 CSV Path: {Config.CAPTIONS_CSV}")
print(f"🤗 HF Dataset: {Config.DATASET_NAME}")
print(f"📏 Sample Size: {Config.SAMPLE_SIZE}")

try:
    # Load and merge datasets
    print("\n🔗 Loading and merging datasets...")
    success = data_manager.load_and_merge_datasets()
    
    if not success:
        print("❌ Data loading failed!")
        print("\n🔧 Troubleshooting:")
        print("1. Check if CSV file exists at specified path")
        print("2. Check internet connection for HuggingFace dataset")
        print("3. Verify CSV file format (should have 'title' and 'caption' columns)")
    else:
        # Prepare data
        print("\n🧹 Preparing and cleaning data...")
        data_manager.prepare_data()
        
        # Show data sample
        print("\n🔍 Data sample:")
        print(data_manager.df[['title', 'caption', '2_way_label']].head())
        
        print("\n✅ Data loading and preparation complete!")
        print(f"📊 Final dataset size: {len(data_manager.df)}")
        print(f"📋 Memory usage: {data_manager.df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
        
except Exception as e:
    print(f"❌ Data preparation error: {e}")
    import traceback
    traceback.print_exc()
    print("\n💡 Tips:")
    print("- Make sure your CSV file is in the correct location")
    print("- Check that the CSV has the required columns: 'title', 'caption'")
    print("- Verify internet connection for downloading HuggingFace dataset")

📊 Starting data loading and preparation...
📄 CSV Path: /kaggle/input/fakeddit-captions-optimized/fakeddit_captions_optimized.csv
🤗 HF Dataset: rtfarchitect/fakeddit_sample
📏 Sample Size: 100000

🔗 Loading and merging datasets...
📊 Loading Fakeddit dataset: rtfarchitect/fakeddit_sample
✅ Loaded HuggingFace dataset: DatasetDict({
    train: Dataset({
        features: ['author', 'clean_title', 'created_utc', 'domain', 'hasImage', 'id', 'image_url', 'linked_submission_id', 'num_comments', 'score', 'subreddit', 'title', 'upvote_ratio', '2_way_label', '3_way_label', '6_way_label'],
        num_rows: 100000
    })
})
Fakeddit dataset shape: (100000, 16)
📄 Loading captions CSV: /kaggle/input/fakeddit-captions-optimized/fakeddit_captions_optimized.csv
✅ Loaded captions CSV
Captions CSV shape: (100000, 2)
🔗 Merging datasets...
✅ Merged dataset created!
Final dataset shape: (100000, 4)

🧹 Preparing and cleaning data...
Using all 100000 records for training
Removed 980 samples with short titles
✅

In [12]:
"""
Extract features using GPU-optimized multi-BERT ensemble and ViT
⚠️ This is the most GPU-intensive part - may take 30-60 minutes!
Monitor GPU memory and reduce batch sizes if needed.
"""

print("🔥 Starting GPU-optimized feature extraction...")
print("⚠️  This may take 30-60 minutes depending on dataset size and GPU")
print(f"📊 Processing {len(data_manager.df)} samples")
print(f"🔧 Text batch size: {Config.FEATURE_BATCH_SIZE}")
print(f"🔧 Image batch size: {Config.FEATURE_BATCH_SIZE // 4}")
print(f"🚀 Initial GPU Memory: {get_gpu_memory_info()}")

try:
    # Extract features
    data_manager.extract_features()
    
    print("\n✅ Feature extraction completed successfully!")
    print(f"📏 Text embeddings: {data_manager.text_embs.shape}")
    print(f"📏 Image embeddings: {data_manager.img_embs.shape}")
    print(f"🚀 Final GPU Memory: {get_gpu_memory_info()}")
    
    # Verify embeddings
    print("\n🔍 Embedding verification:")
    print(f"Text embeddings - Min: {data_manager.text_embs.min():.3f}, Max: {data_manager.text_embs.max():.3f}")
    print(f"Image embeddings - Min: {data_manager.img_embs.min():.3f}, Max: {data_manager.img_embs.max():.3f}")
    
    # Check for NaN values
    text_nan = torch.isnan(data_manager.text_embs).sum().item()
    img_nan = torch.isnan(data_manager.img_embs).sum().item()
    print(f"NaN values - Text: {text_nan}, Images: {img_nan}")
    
    if text_nan > 0 or img_nan > 0:
        print("⚠️  Found NaN values in embeddings - this may affect training")
    
    print("\n🎉 Ready for data splitting and training!")
    
except Exception as e:
    print(f"❌ Feature extraction error: {e}")
    import traceback
    traceback.print_exc()
    
    print("\n🔧 Troubleshooting:")
    print("1. Reduce Config.FEATURE_BATCH_SIZE (try 16 or 8)")
    print("2. Set Config.USE_MIXED_PRECISION = False")
    print("3. Reduce Config.SAMPLE_SIZE (try 10000)")
    print("4. Restart notebook kernel to free GPU memory")
    
    # Show current GPU memory for debugging
    print(f"\n🚀 Current GPU Memory: {get_gpu_memory_info()}")
    
    # Clean up memory
    optimize_gpu_memory()

🔥 Starting GPU-optimized feature extraction...
⚠️  This may take 30-60 minutes depending on dataset size and GPU
📊 Processing 99020 samples
🔧 Text batch size: 32
🔧 Image batch size: 8
🚀 Initial GPU Memory: {'allocated': 0.8871989250183105, 'cached': 1.86328125, 'max_allocated': 1.7311859130859375}
🔥 Starting GPU-optimized feature extraction...
🚀 Initial GPU Memory: {'allocated': 0.8871989250183105, 'cached': 1.86328125, 'max_allocated': 1.7311859130859375}
🔥 Starting GPU-optimized ensemble text embedding extraction...


GPU Batch Text Processing:   0%|          | 0/3095 [00:00<?, ?it/s]

🎯 Final text embeddings shape: torch.Size([99020, 768])
🚀 GPU Memory after text extraction: {'allocated': 1.4626879692077637, 'cached': 2.123046875, 'max_allocated': 1.7311859130859375}
🔥 Starting GPU-optimized image embedding extraction...


GPU Batch Image Processing:   0%|          | 0/12378 [00:00<?, ?it/s]

⚠️  488/99020 images failed to load (using zero embeddings)
🎯 Final image embeddings shape: torch.Size([99020, 512])
🚀 GPU Memory after image extraction: {'allocated': 1.4661717414855957, 'cached': 2.12109375, 'max_allocated': 1.7311859130859375}
✅ GPU-optimized feature extraction complete!
Text embeddings shape: torch.Size([99020, 768])
Image embeddings shape: torch.Size([99020, 512])
🚀 Final GPU Memory: {'allocated': 1.3687396049499512, 'cached': 2.12109375, 'max_allocated': 1.7311859130859375}

✅ Feature extraction completed successfully!
📏 Text embeddings: torch.Size([99020, 768])
📏 Image embeddings: torch.Size([99020, 512])
🚀 Final GPU Memory: {'allocated': 1.3687396049499512, 'cached': 2.0234375, 'max_allocated': 1.7311859130859375}

🔍 Embedding verification:
Text embeddings - Min: -6.092, Max: 5.436
Image embeddings - Min: -10.859, Max: 5.488
NaN values - Text: 0, Images: 0

🎉 Ready for data splitting and training!


In [13]:
"""
Split data and create CUDA-safe DataLoaders for training
This cell prepares everything for model training
"""

print("📋 Splitting data and creating DataLoaders...")
print(f"🎯 Train split: {Config.TRAIN_SPLIT * 100:.0f}%")
print(f"🎯 Validation split: {Config.VAL_SPLIT * 100:.0f}%")
print(f"🎯 Test split: {Config.TEST_SPLIT * 100:.0f}%")

try:
    # Split data
    train_idx, val_idx, test_idx = data_manager.split_data()
    
    # Create DataLoaders
    print("\n🔧 Creating CUDA-safe DataLoaders...")
    train_loader, val_loader, test_loader = data_manager.create_dataloaders(
        train_idx, val_idx, test_idx, device
    )
    
    print("\n✅ DataLoaders created successfully!")
    print(f"🔧 Batch size: {Config.BATCH_SIZE}")
    print(f"🔧 Number of workers: {Config.NUM_WORKERS} (CUDA-safe)")
    print(f"🔧 Pin memory: {Config.PIN_MEMORY} (CUDA-safe)")
    
    # Show batch information
    print(f"\n📊 Batch information:")
    print(f"Train batches: {len(train_loader)}")
    print(f"Validation batches: {len(val_loader)}")
    print(f"Test batches: {len(test_loader)}")
    
    # Test a sample batch to ensure everything works
    print("\n🧪 Testing sample batch...")
    sample_batch = next(iter(train_loader))
    text_batch, img_batch, label_batch = sample_batch
    print(f"✅ Sample batch shapes:")
    print(f"   Text: {text_batch.shape}")
    print(f"   Images: {img_batch.shape}")
    print(f"   Labels: {label_batch.shape}")
    print(f"   Device: {text_batch.device}")
    
    # Show label distribution in batches
    print(f"\n📈 Label distribution in sample batch:")
    unique, counts = torch.unique(label_batch, return_counts=True)
    for label, count in zip(unique.cpu().numpy(), counts.cpu().numpy()):
        label_name = "Fake" if label == 1 else "Real"
        print(f"   {label_name}: {count} samples")
    
    print("\n🎉 Data splitting complete! Ready for model initialization and training.")
    
except Exception as e:
    print(f"❌ Data splitting error: {e}")
    import traceback
    traceback.print_exc()
    
    print("\n🔧 This usually indicates a problem with feature extraction.")
    print("Make sure Cell 11 (feature extraction) completed successfully.")

📋 Splitting data and creating DataLoaders...
🎯 Train split: 70%
🎯 Validation split: 15%
🎯 Test split: 15%
✅ Train: 69314 samples
✅ Validation: 14853 samples
✅ Test: 14853 samples

🔧 Creating CUDA-safe DataLoaders...
✅ CUDA-safe DataLoaders created!

✅ DataLoaders created successfully!
🔧 Batch size: 16
🔧 Number of workers: 0 (CUDA-safe)
🔧 Pin memory: False (CUDA-safe)

📊 Batch information:
Train batches: 4333
Validation batches: 929
Test batches: 929

🧪 Testing sample batch...
✅ Sample batch shapes:
   Text: torch.Size([16, 768])
   Images: torch.Size([16, 512])
   Labels: torch.Size([16])
   Device: cuda:0

📈 Label distribution in sample batch:
   Real: 10 samples
   Fake: 6 samples

🎉 Data splitting complete! Ready for model initialization and training.


In [14]:
"""
Initialize the advanced model and start training
This is where the magic happens!
"""

print("🏗️  Initializing advanced multimodal model...")

try:
    # Initialize model
    model = AdvancedMultimodalClassifier().to(device)
    
    # Enable mixed precision for the model if specified
    if Config.USE_MIXED_PRECISION:
        model = model.half()
        print("✅ Model converted to FP16 for mixed precision training")
    
    # Show model information
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"\n🔧 Model Architecture:")
    print(f"   Total parameters: {total_params:,}")
    print(f"   Trainable parameters: {trainable_params:,}")
    print(f"   Model size: ~{total_params * 4 / 1024**2:.1f} MB (FP32)")
    print(f"   Mixed precision: {Config.USE_MIXED_PRECISION}")
    
    # Test model with sample batch
    print("\n🧪 Testing model forward pass...")
    model.eval()
    with torch.no_grad():
        if Config.USE_MIXED_PRECISION:
            with autocast():
                logits, consistency_score, attention_weights = model(text_batch, img_batch)
        else:
            logits, consistency_score, attention_weights = model(text_batch, img_batch)
    
    print(f"✅ Forward pass successful!")
    print(f"   Logits shape: {logits.shape}")
    print(f"   Consistency score shape: {consistency_score.shape}")
    print(f"   Attention weights shape: {attention_weights.shape}")
    
    print(f"\n🚀 Model GPU Memory: {get_gpu_memory_info()}")
    
    # Initialize trainer
    print("\n🎯 Initializing advanced trainer...")
    trainer = AdvancedTrainer(model, device)
    
    print("\n🔥 Starting training!")
    print(f"📊 Training settings:")
    print(f"   Max epochs: {Config.MAX_EPOCHS}")
    print(f"   Patience: {Config.PATIENCE}")
    print(f"   Learning rate: {Config.LEARNING_RATE}")
    print(f"   Batch size: {Config.BATCH_SIZE}")
    print(f"   Mixed precision: {Config.USE_MIXED_PRECISION}")
    
    # Start training
    trainer.train(train_loader, val_loader)
    
    print("\n🎉 Training completed! Model saved.")
    print(f"💾 Best model saved at: {Config.BEST_MODEL_PATH}")
    
except Exception as e:
    print(f"❌ Training error: {e}")
    import traceback
    traceback.print_exc()
    
    print("\n🔧 Training troubleshooting:")
    print("1. Reduce Config.BATCH_SIZE (try 8 or 4)")
    print("2. Set Config.USE_MIXED_PRECISION = False")
    print("3. Reduce Config.HIDDEN_DIM (try 512)")
    print("4. Check if previous cells completed successfully")
    
    # Memory cleanup
    optimize_gpu_memory()
    print(f"\n🚀 GPU Memory after cleanup: {get_gpu_memory_info()}")

🏗️  Initializing advanced multimodal model...
✅ Model converted to FP16 for mixed precision training

🔧 Model Architecture:
   Total parameters: 10,374,918
   Trainable parameters: 10,374,918
   Model size: ~39.6 MB (FP32)
   Mixed precision: True

🧪 Testing model forward pass...
✅ Forward pass successful!
   Logits shape: torch.Size([16, 2])
   Consistency score shape: torch.Size([16, 1])
   Attention weights shape: torch.Size([16, 8, 1, 1])

🚀 Model GPU Memory: {'allocated': 1.8622956275939941, 'cached': 2.599609375, 'max_allocated': 2.006166934967041}

🎯 Initializing advanced trainer...
🔥 Trainer initialized with mixed precision: True
🚀 GPU Memory after trainer init: {'allocated': 1.8622956275939941, 'cached': 2.599609375, 'max_allocated': 2.006166934967041}

🔥 Starting training!
📊 Training settings:
   Max epochs: 50
   Patience: 10
   Learning rate: 5e-05
   Batch size: 16
   Mixed precision: True
🔥 Starting GPU-optimized mixed precision training...
🚀 Training GPU Memory: {'alloca

🔥 Mixed Precision Training:   0%|          | 0/4333 [00:00<?, ?it/s]

❌ Training error: torch.nn.functional.binary_cross_entropy and torch.nn.BCELoss are unsafe to autocast.
Many models use a sigmoid layer right before the binary cross entropy layer.
In this case, combine the two layers using torch.nn.functional.binary_cross_entropy_with_logits
or torch.nn.BCEWithLogitsLoss.  binary_cross_entropy_with_logits and BCEWithLogits are
safe to autocast.

🔧 Training troubleshooting:
1. Reduce Config.BATCH_SIZE (try 8 or 4)
2. Set Config.USE_MIXED_PRECISION = False
3. Reduce Config.HIDDEN_DIM (try 512)
4. Check if previous cells completed successfully

🚀 GPU Memory after cleanup: {'allocated': 1.8635902404785156, 'cached': 2.59765625, 'max_allocated': 2.006166934967041}


Traceback (most recent call last):
  File "/tmp/ipykernel_296/2435697154.py", line 57, in <cell line: 0>
    trainer.train(train_loader, val_loader)
  File "/tmp/ipykernel_296/2677179825.py", line 137, in train
    avg_train_loss = self.train_epoch(train_loader)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_296/2677179825.py", line 54, in train_epoch
    consistency_loss = F.binary_cross_entropy(consistency_score.squeeze(), consistency_target)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/nn/functional.py", line 3569, in binary_cross_entropy
    return torch._C._nn.binary_cross_entropy(input, target, weight, reduction_enum)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: torch.nn.functional.binary_cross_entropy and torch.nn.BCELoss are unsafe to autocast.
Many models use a sigmoid layer right before the bin

In [15]:
"""
Evaluate the trained model and show comprehensive results
This is the final step - see how well your model performed!
"""

print("📈 Starting model evaluation...")

try:
    # Initialize evaluator
    evaluator = AdvancedEvaluator(model, device)
    
    # Perform evaluation
    print("\n🎯 Evaluating on test set...")
    test_acc, test_preds, test_labels = evaluator.evaluate(test_loader)
    
    # Additional analysis
    print("\n📊 Additional Analysis:")
    
    # Per-class accuracy
    from sklearn.metrics import accuracy_score
    real_mask = np.array(test_labels) == 0
    fake_mask = np.array(test_labels) == 1
    
    real_acc = accuracy_score(np.array(test_labels)[real_mask], np.array(test_preds)[real_mask])
    fake_acc = accuracy_score(np.array(test_labels)[fake_mask], np.array(test_preds)[fake_mask])
    
    print(f"📈 Per-class accuracy:")
    print(f"   Real news: {real_acc:.4f} ({real_acc*100:.2f}%)")
    print(f"   Fake news: {fake_acc:.4f} ({fake_acc*100:.2f}%)")
    
    # Show some prediction examples
    print(f"\n🔍 Sample predictions:")
    sample_indices = np.random.choice(len(test_labels), 5, replace=False)
    
    for i, idx in enumerate(sample_indices):
        true_label = "Fake" if test_labels[idx] == 1 else "Real"
        pred_label = "Fake" if test_preds[idx] == 1 else "Real"
        correct = "✅" if test_labels[idx] == test_preds[idx] else "❌"
        
        print(f"   Example {i+1}: True={true_label}, Pred={pred_label} {correct}")
    
    # Model performance summary
    print(f"\n🏆 FINAL PERFORMANCE SUMMARY:")
    print(f"=" * 50)
    print(f"🎯 Overall Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
    print(f"📊 Total test samples: {len(test_labels)}")
    print(f"✅ Correct predictions: {sum(np.array(test_preds) == np.array(test_labels))}")
    print(f"❌ Wrong predictions: {sum(np.array(test_preds) != np.array(test_labels))}")
    print(f"🚀 Peak GPU Memory: {get_gpu_memory_info()['max_allocated']:.1f}GB")
    print(f"=" * 50)
    
    # Save results
    results = {
        'test_accuracy': test_acc,
        'test_predictions': test_preds,
        'test_labels': test_labels,
        'real_accuracy': real_acc,
        'fake_accuracy': fake_acc,
        'model_config': {
            'batch_size': Config.BATCH_SIZE,
            'learning_rate': Config.LEARNING_RATE,
            'mixed_precision': Config.USE_MIXED_PRECISION,
            'hidden_dim': Config.HIDDEN_DIM
        }
    }
    
    # Save to file
    import pickle
    results_path = os.path.join(Config.RESULTS_DIR, 'evaluation_results.pkl')
    with open(results_path, 'wb') as f:
        pickle.dump(results, f)
    
    print(f"\n💾 Results saved to: {results_path}")
    print(f"💾 Model saved at: {Config.BEST_MODEL_PATH}")
    
    print("\n🎉 Evaluation completed successfully!")
    print("\n📋 What's Next:")
    print("1. Check the saved model for deployment")
    print("2. Analyze attention weights for interpretability")
    print("3. Try different hyperparameters for improvement")
    print("4. Test on new data")
    
except Exception as e:
    print(f"❌ Evaluation error: {e}")
    import traceback
    traceback.print_exc()
    
    print("\n🔧 This usually means training didn't complete successfully.")
    print("Make sure Cell 13 (training) completed without errors.")
    print(f"Check if model file exists: {Config.BEST_MODEL_PATH}")

# Final cleanup
optimize_gpu_memory()
print(f"\n🧹 Final GPU Memory: {get_gpu_memory_info()}")

📈 Starting model evaluation...

🎯 Evaluating on test set...
🔥 Starting GPU-optimized evaluation...
❌ Evaluation error: [Errno 2] No such file or directory: '/kaggle/working/trained_models/best_advanced_model.pth'

🔧 This usually means training didn't complete successfully.
Make sure Cell 13 (training) completed without errors.
Check if model file exists: /kaggle/working/trained_models/best_advanced_model.pth


Traceback (most recent call last):
  File "/tmp/ipykernel_296/1380691378.py", line 14, in <cell line: 0>
    test_acc, test_preds, test_labels = evaluator.evaluate(test_loader)
                                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_296/2677179825.py", line 193, in evaluate
    checkpoint = torch.load(Config.BEST_MODEL_PATH)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/serialization.py", line 1425, in load
    with _open_file_like(f, "rb") as opened_file:
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/serialization.py", line 751, in _open_file_like
    return _open_file(name_or_buffer, mode)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/serialization.py", line 732, in __init__
    super().__init__(open(name, mode))
                     ^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or


🧹 Final GPU Memory: {'allocated': 1.8622956275939941, 'cached': 2.59765625, 'max_allocated': 2.006166934967041}
